```bash
═════════════════════════════════════════════════════════════════════════════════════════
Filtro Small Caps Target Population (< $2B) con Anti-Survivorship Bias
═════════════════════════════════════════════════════════════════════════════════════════
Definir small-cap en forma point-in-time (sin look-ahead bias).

Regla por fecha t:
- market_cap_t = close_t * shares_outstanding_t
- close_t viene de day_aggs_v1
- small_cap_t = market_cap_t < 2_000_000_000

Fallback obligatorio si falta shares_outstanding_t:
- usar último valor válido anterior (LOCF) con límite máximo de antigüedad (ejemplo: 180 días).
- si no hay valor dentro del límite: market_cap_t = null e is_small_cap_t = null.
- nunca imputar con datos futuros.

Notas críticas:
- No clasificar 2005-2026 con market_cap "actual".
- Mantener inactivos históricos que fueron small-cap en su ventana de vida.
- Guardar tabla final:
- population_target_pti(date, figi, ticker, is_small_cap, exchange, status).

#####################################################################
# Problema real y solucion --> Necesitamos datos de los fundamentales.
#####################################################################

Validé que:

- el script actual sí soporta --resume-validate y --progress-every;
- --help refleja exactamente esos flags;
- tu input tiene 12,473 tickers únicos, así que procesará aprox. 12,473 x 4 = 49,892 tareas dataset/ticker.

python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\download_fundamentals_v1.py 
   --input C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026.parquet 
   --outdir D:\financial --datasets income_statements,balance_sheets,cash_flow_statements,ratios 
   --batch-size 200 
   --workers 12 
   --limit 1000 
   --max-pages 50 
   --resume 
   --resume-validate 
   --progress-every 25

ticker 10/12473 (0.08%) last=ratios:AAGR rows=1 status=200 eta=2388s
ticker 20/12473 (0.16%) last=ratios:AAP rows=1 status=200 eta=1977s
...
...
ticker 12470/12473 (99.98%) last=ratios:ZYME rows=1 status=200 eta=0s
ticker 12473/12473 (100.00%) last=ratios:ZZ rows=0 status=200 eta=0s
tickers_processed=12,473
tickers_skipped_valid=0
outdir=D:\financial
progress=D:\financial\_run\download_fundamentals_v1.progress.json
errors=D:\financial\_run\download_fundamentals_v1.errors.csv
```

In [ ]:
python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\01_data_fundamentals\cell_code\audit_fundamentals_download.py ` --input-universe C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper.parquet ` --outdir D:\financial ` --datasets income_statements,balance_sheets,cash_flow_statements,ratios `
    --limit 1000 `
    --max-pages 50 `
    --lifecycle-parquet C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\official_lifecycle_compiled.multisource.parquet `
    --tolerance-pre-days 365 `
    --tolerance-post-days 180 `
    --audit-dir C:\Users\AlexJ\financial_audit



import pandas as pd
p=r'D:\financial\_run\download_fundamentals_v1.errors.csv'
d=pd.read_csv(p)
print('rows',len(d))
print('http_status',d['http_status'].value_counts(dropna=False).to_dict())
print('top_msg',d['msg'].value_counts(dropna=False).head(10).to_dict())

rows 263
http_status {200: 263}
top_msg {'ok | dropped_mismatch=5198 dropped_missing_ticker=0': 151, 'ok | dropped_mismatch=5199 dropped_missing_ticker=0': 110, 'ok | dropped_mismatch=5197 dropped_missing_ticker=0': 2}


In [22]:
# Celda 1: cargar una tabla concreta (elige solo una)
import pandas as pd
from pathlib import Path
from IPython.display import display

PATHS = {
    "balance_sheets": Path(r"D:\financial\balance_sheets\ticker=A\balance_sheets_A.parquet"),
    "cash_flow_statements": Path(r"D:\financial\cash_flow_statements\ticker=A\cash_flow_statements_A.parquet"),
    "income_statements": Path(r"D:\financial\income_statements\ticker=A\income_statements_A.parquet"),
    "ratios": Path(r"D:\financial\ratios\ticker=A\ratios_A.parquet"),
}

In [23]:
# cambia: balance_sheets | cash_flow_statements | income_statements | ratios
p = PATHS["balance_sheets"]
df = pd.read_parquet(p)
print("->", p)
print("shape:", df.shape)
display(df.head(2).T)
print(df.columns)

-> D:\financial\balance_sheets\ticker=A\balance_sheets_A.parquet
shape: (80, 40)


,0,1
tickers,[A],[A]
cik,0001090872,0001090872
period_end,2010-04-30,2010-07-31
filing_date,2010-06-07,2010-10-06
fiscal_quarter,2,3
fiscal_year,2010,2010
timeframe,quarterly,quarterly
cash_and_equivalents,2646000000.0,2317000000.0
short_term_investments,11000000.0,NaN
receivables,669000000.0,790000000.0


Index(['tickers', 'cik', 'period_end', 'filing_date', 'fiscal_quarter',
       'fiscal_year', 'timeframe', 'cash_and_equivalents',
       'short_term_investments', 'receivables', 'inventories',
       'other_current_assets', 'total_current_assets',
       'property_plant_equipment_net', 'goodwill', 'intangible_assets_net',
       'other_assets', 'total_assets', 'accounts_payable',
       'deferred_revenue_current', 'debt_current',
       'accrued_and_other_current_liabilities', 'total_current_liabilities',
       'long_term_debt_and_capital_lease_obligations',
       'other_noncurrent_liabilities', 'total_liabilities', 'common_stock',
       'additional_paid_in_capital', 'treasury_stock',
       'accumulated_other_comprehensive_income', 'retained_earnings_deficit',
       'other_equity', 'total_equity_attributable_to_parent',
       'noncontrolling_interest', 'total_equity',
       'total_liabilities_and_equity', 'preferred_stock', 'ticker', '_dataset',
       '_ingested_utc'],
      d

In [24]:
# cambia: balance_sheets | cash_flow_statements | income_statements | ratios
p = PATHS["cash_flow_statements"]
df = pd.read_parquet(p)
print(df.columns)
print("->", p)
print("shape:", df.shape)
display(df.head(2).T)

Index(['tickers', 'cik', 'period_end', 'filing_date', 'fiscal_quarter',
       'fiscal_year', 'timeframe', 'net_income',
       'depreciation_depletion_and_amortization', 'other_operating_activities',
       'change_in_other_operating_assets_and_liabilities_net',
       'cash_from_operating_activities_continuing_operations',
       'net_cash_from_operating_activities',
       'purchase_of_property_plant_and_equipment',
       'other_investing_activities',
       'net_cash_from_investing_activities_continuing_operations',
       'net_cash_from_investing_activities',
       'long_term_debt_issuances_repayments', 'dividends',
       'other_financing_activities',
       'net_cash_from_financing_activities_continuing_operations',
       'net_cash_from_financing_activities', 'change_in_cash_and_equivalents',
       'effect_of_currency_exchange_rate',
       'sale_of_property_plant_and_equipment', 'noncontrolling_interests',
       'short_term_debt_issuances_repayments', 'ticker', '_dataset',

,0,1
tickers,[A],[A]
cik,0001090872,0001090872
period_end,2010-01-31,2010-04-30
filing_date,2011-03-09,2011-06-07
fiscal_quarter,1,2
fiscal_year,2010,2010
timeframe,quarterly,quarterly
net_income,79000000.0,108000000.0
depreciation_depletion_and_amortization,39000000.0,36000000.0
other_operating_activities,134000000.0,-18000000.0


In [26]:
# cambia: balance_sheets | cash_flow_statements | income_statements | ratios
p = PATHS["income_statements"]
df = pd.read_parquet(p)
print(df.columns)
print("->", p)
print("shape:", df.shape)
display(df.head(2).T)

Index(['tickers', 'cik', 'period_end', 'filing_date', 'fiscal_quarter',
       'fiscal_year', 'timeframe', 'revenue', 'cost_of_revenue',
       'gross_profit', 'selling_general_administrative',
       'research_development', 'other_operating_expenses',
       'total_operating_expenses', 'operating_income', 'interest_expense',
       'interest_income', 'other_income_expense', 'total_other_income_expense',
       'income_before_income_taxes', 'income_taxes',
       'consolidated_net_income_loss',
       'net_income_loss_attributable_common_shareholders',
       'basic_earnings_per_share', 'diluted_earnings_per_share',
       'basic_shares_outstanding', 'diluted_shares_outstanding', 'ebitda',
       'discontinued_operations', 'ticker', '_dataset', '_ingested_utc'],
      dtype='str')
-> D:\financial\income_statements\ticker=A\income_statements_A.parquet
shape: (143, 32)


,0,1
tickers,[A],[A]
cik,0001090872,0001090872
period_end,2010-01-31,2010-04-30
filing_date,2011-03-09,2011-06-07
fiscal_quarter,1,2
fiscal_year,2010,2010
timeframe,quarterly,quarterly
revenue,1213000000.0,1271000000.0
cost_of_revenue,553000000.0,560000000.0
gross_profit,660000000.0,711000000.0


In [28]:
# cambia: balance_sheets | cash_flow_statements | income_statements | ratios
p = PATHS["ratios"]
df = pd.read_parquet(p)
print(df.columns)
print("->", p)
print("shape:", df.shape)
display(df.head(2).T)

Index(['ticker', 'cik', 'date', 'price', 'average_volume', 'market_cap',
       'earnings_per_share', 'price_to_earnings', 'price_to_book',
       'price_to_sales', 'price_to_cash_flow', 'price_to_free_cash_flow',
       'dividend_yield', 'return_on_assets', 'return_on_equity',
       'debt_to_equity', 'current', 'quick', 'cash', 'ev_to_sales',
       'ev_to_ebitda', 'enterprise_value', 'free_cash_flow', '_dataset',
       '_ingested_utc'],
      dtype='str')
-> D:\financial\ratios\ticker=A\ratios_A.parquet
shape: (1, 25)


,0
ticker,A
cik,0001090872
date,2026-03-06
price,115.07
average_volume,2424738.0
market_cap,32519048617.0
earnings_per_share,4.56
price_to_earnings,25.21
price_to_book,4.71
price_to_sales,4.6


## Queremos saber listed, deslisted

Con lo que tienes hoy, la comparación más segura se hace así:

**1. Fuente de verdad de lifecycle**

- Usa solo:
    - official_lifecycle_compiled.multisource.parquet
- No uses list_date/delisted_utc de universe_pti porque están vacíos ahora.

**2. Cobertura real descargada**

- Para cada ticker en D:\financial calcula:
    - fund_min_date
    - fund_max_date
- Fechas por dataset:
    - income/balance/cash: period_end (fallback filing_date)
    - ratios: date

**3. Join robusto**

- Join principal por ticker (uppercase, trim).
- Si hay cik en fundamentals y lifecycle, exige también coherencia de cik cuando ambos existan (esto evita errores por ticker
reciclado).

**4. Reglas de validación**

- delist_date no nula:
    - fund_max_date <= delist_date + 180d (gracia por retraso de filing).
- list_date no nula:
    - fund_min_date >= list_date - 365d (gracia por estructura fiscal/reporting).
- Si rompe regla: ANOMALY.
- Si faltan fechas lifecycle: UNKNOWN.
- Si no hay fundamentals para ticker: NO_DATA.

**5. Salida certificable**

- Genera un reporte único con columnas:
    - ticker, cik_lifecycle, cik_fund, list_date, delist_date, fund_min_date, fund_max_date, status, reason
- status en: OK, ANOMALY_POST_DELIST, ANOMALY_PRE_LIST, CIK_MISMATCH, NO_DATA, UNKNOWN.

**6. “100% segura” (realista)**

- Con ticker solo no existe 100% matemático por reutilización de símbolos.
- Para máxima seguridad: prioriza matching por CIK/FIGI/entity_id y deja ticker como secundario.

Si quieres, te preparo el script de auditoría final (scripts/agent_fundamentals_lifecycle_audit.py) con ese reporte y umbrales
configurables.

Así auditamos con “certeza operativa 100%” (determinista, no heurística):

  1. Cobertura completa por endpoint

  - Universo esperado = tickers únicos de tickers_2005_2026.parquet.
  - Para cada endpoint, exige que exista archivo para cada ticker.
  - Reporta missing_tickers, extra_tickers y tickers con múltiples archivos.

  2. Integridad file-by-file (contenido)

  - Cada parquet debe abrir sin error.
  - ticker interno debe ser único y coincidir con carpeta ticker=....
  - Si existe tickers, valida que contenga el ticker esperado.
  - Detecta mezcla por CIK (cik_nunique > 1).
  - Marca truncado/sobredescarga si filas alcanzan limit*max_pages.

  3. Rango temporal por ticker+endpoint

  - Crea date_start y date_end por archivo con candidatos mejorados por dataset:
      - statements: period_end, filing_date, _ingested_utc
      - ratios: date, _ingested_utc
  - Guarda tabla final por ticker+endpoint con esas columnas.

  Ya te lo dejé aquí:
  C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\01_data_fundamentals\cell_code\audit_fundamentals_download.py

  Lanzadera terminal:

  python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\01_data_fundamentals\cell_code\audit_fundamentals_download.py --input-universe C:
  \TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026.parquet --outdir D:\financial --datasets
  income_statements,balance_sheets,cash_flow_statements,ratios --limit 1000 --max-pages 50

  Outputs:

  - D:\financial\_audit\audit_summary.json
  - D:\financial\_audit\coverage_by_endpoint.csv
  - D:\financial\_audit\missing_tickers_by_endpoint.csv
  - D:\financial\_audit\file_level_audit.csv
  - D:\financial\_audit\date_ranges_by_ticker_endpoint.csv
  - D:\financial\_audit\severe_issues.csv

================================================================
```bash
- completed 12473/12473: ok proceso.
- http_status=200 con dropped_mismatch: no es error de red; es trazabilidad de filtro.
- Integridad archivo/ticker en muestra: buena señal.
- 12468 carpetas vs 12473 tickers: bloqueante de cobertura por colisión case-insensitive en Windows.

Qué haría ya (mínimo necesario):

1. Normalizar input a UPPER() y deduplicar antes de descargar.
2. Relanzar con --resume --resume-validate para converger a cardinalidad real.
3. Separar en auditoría:

- errors_http (status != 200/404)
- warnings_filter (dropped_mismatch / dropped_missing_ticker)

Comando rápido para generar input normalizado:

python -c "import pandas as pd; p=r'C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026.parquet'; o=r'C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper.parquet'; d=pd.read_parquet(p); d['ticker']=d['ticker'].astype('string').str.strip().str.upper(); d=d.dropna(subset=['ticker']).drop_duplicates(subset=['ticker']).sort_values('ticker'); d.to_parquet(o,index=False); print('rows',len(d)); print('saved',o)"

Luego relanza usando ese input:

python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\download_fundamentals_v1.py 
   --input C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper.parquet 
   --outdir D:\financial 
   --datasets income_statements,balance_sheets,cash_flow_statements,ratios 
   --batch-size 200 
   --workers 12 
   --limit 1000 
   --max-pages 50 
   --resume 
   --resume-validate 
   --progress-every 25

# resultado
ticker 10/263 (3.80%) last=ratios:AHPA rows=0 status=200 eta=45s
ticker 263/263 (100.00%) last=ratios:XPER rows=1 status=200 eta=0s
tickers_processed=263
tickers_skipped_valid=12,205
outdir=D:\financial
progress=D:\financial\_run\download_fundamentals_v1.progress.json
errors=D:\financial\_run\download_fundamentals_v1.errors.csv
PS C:\Users\AlexJ>

## Auditoria de la descarga 

`C:\Users\AlexJ\financial_audit`

**Este agente a analizado**

1. Cobertura por dataset

- expected_tickers vs downloaded_tickers
- missing_tickers, extra_tickers
- tickers_with_multiple_files

2. Integridad de archivo por ticker+dataset

- archivo legible (read_ok)
- rows_total, rows_business (excluyendo _empty=True)
- ticker canónico correcto (ticker_col_ok)
- mismatches en campo tickers (tickers_col_mismatch_rows)
- unicidad de cik (cik_nunique)
- columna _dataset consistente (dataset_col_ok)
- parseo de _ingested_utc (ingested_utc_parseable)
- columnas mínimas requeridas (missing_required_cols)
- señal de truncado por paginación (rows_hit_page_cap vía limit*max_pages)

3. Rango temporal por archivo

- detecta columna de fecha usada (date_col_used)
- calcula date_start, date_end por archivo

4. Validación temporal por ticker (global)

- consolida fin_min_date / fin_max_date (todos los datasets)
- construye ventana de referencia:
    - prioridad lifecycle oficial (official_list_date, official_delist_date)
    - fallback PTI (first_seen_date, last_seen_date)
- compara con tolerancias (--tolerance-pre-days, --tolerance-post-days)
- etiqueta estado por ticker:
    - OK
    - NO_DATA
    - ANOMALY_PRE_START
    - ANOMALY_POST_END
    - UNKNOWN_WINDOW

5. Reportes que genera

- coverage_by_endpoint.csv
- missing_tickers_by_endpoint.csv
- file_level_audit.csv
- date_ranges_by_ticker_endpoint.csv
- severe_issues.csv
- temporal_validation_by_ticker.csv
- temporal_issues.csv
- audit_summary.json

In [69]:
import pandas as pd
from pathlib import Path

# Auditoría global
p_audit = Path(r"C:\Users\AlexJ\financial_audit\file_level_audit.csv")
d = pd.read_csv(p_audit)

# Archivo ejemplo endpoint income_statements
p_income = Path(r"D:\financial\income_statements\ticker=A\income_statements_A.parquet")
df = pd.read_parquet(p_income)

def print_dataset_report(df_audit: pd.DataFrame, ds: str):
    g = df_audit[df_audit["dataset"] == ds].copy()
    print(f"\n=== {ds} ===")
    print("files:", len(g))
    print("read_ok:", g["read_ok"].value_counts(dropna=False).to_dict())
    print("ticker_col_ok:", g["ticker_col_ok"].value_counts(dropna=False).to_dict())
    print("dataset_col_ok:", g["dataset_col_ok"].value_counts(dropna=False).to_dict())
    print("ingested_utc_parseable:", g["ingested_utc_parseable"].value_counts(dropna=False).to_dict())
    print("rows_business_zero:", int((g["rows_business"].fillna(0) == 0).sum()))
    s = g["tickers_col_mismatch_rows"].fillna(0)
    print("tickers_col_mismatch_rows_gt0:", int((s > 0).sum()))
    c = g["cik_nunique"].fillna(0)
    print("cik_nunique:", c.value_counts(dropna=False).sort_index().to_dict())
    mrc = g["missing_required_cols"].fillna("")
    print("missing_required_cols:", mrc.value_counts(dropna=False).to_dict())
    print("rows_hit_page_cap:", int(g["issues"].astype(str).str.contains("rows_hit_page_cap", na=False).sum()))

print("auditando financials INCOME STATEMENTS")
print()
print("->", p_income)
print("\nColumnas:")
print(df.columns)

print("\nAuditoría de la descarga total en: D:\\financial\\income_statements\\")
print_dataset_report(d, "income_statements")



auditando financials INCOME STATEMENTS

-> D:\financial\income_statements\ticker=A\income_statements_A.parquet

Columnas:
Index(['tickers', 'cik', 'period_end', 'filing_date', 'fiscal_quarter',
       'fiscal_year', 'timeframe', 'revenue', 'cost_of_revenue',
       'gross_profit', 'selling_general_administrative',
       'research_development', 'other_operating_expenses',
       'total_operating_expenses', 'operating_income', 'interest_expense',
       'interest_income', 'other_income_expense', 'total_other_income_expense',
       'income_before_income_taxes', 'income_taxes',
       'consolidated_net_income_loss',
       'net_income_loss_attributable_common_shareholders',
       'basic_earnings_per_share', 'diluted_earnings_per_share',
       'basic_shares_outstanding', 'diluted_shares_outstanding', 'ebitda',
       'discontinued_operations', 'ticker', '_dataset', '_ingested_utc'],
      dtype='str')

Auditoría de la descarga total en: D:\financial\income_statements\

=== income_statem

- **files=12468**: cobertura completa del endpoint.
- **read_ok=True 12468**: todos legibles.
- **ticker_col_ok=True 12468**: ticker interno correcto.
- **dataset_col_ok=True 12468**: _dataset consistente.
- **ingested_utc_parseable=True 12468**: timestamp bien formado.
- **rows_business_zero=4443**: sin datos de negocio en ~35.6% de tickers.
- **tickers_col_mismatch_rows_gt0=0**: sin contradicciones en tickers.
- **cik_nunique={0:4443,1:7766,2:249,3:10}:**
    - ideal (1): 7,766
    - sin CIK (0): 4,443
    - ambiguo (>1): 259 archivos (249+10)
- **missing_required_cols={'':8025,'cik':4443}**: coincide con los vacíos.
- **rows_hit_page_cap=0**: sin truncado por paginación.

Conclusión de income_statements: integridad técnica buena, con dos puntos de negocio a vigilar: rows_business_zero y cik_nunique>1
(259 casos).

In [ ]:
print("auditando financials CASH FLOW STATEMENTS")
print()
print("->", p_)
print()
p_ = PATHS["cash_flow_statements"]
df = pd.read_parquet(p_)
print(df.columns)
# display(df.head(1).T)

# 3) cash_flow_statements
print("Auditoria de la descarga total en : financials/cash_flow_statements/")
print_dataset_report(d, "cash_flow_statements")

auditando financials CASH FLOW STATEMENTS

-> D:\financial\cash_flow_statements\ticker=A\cash_flow_statements_A.parquet

Index(['tickers', 'cik', 'period_end', 'filing_date', 'fiscal_quarter',
       'fiscal_year', 'timeframe', 'net_income',
       'depreciation_depletion_and_amortization', 'other_operating_activities',
       'change_in_other_operating_assets_and_liabilities_net',
       'cash_from_operating_activities_continuing_operations',
       'net_cash_from_operating_activities',
       'purchase_of_property_plant_and_equipment',
       'other_investing_activities',
       'net_cash_from_investing_activities_continuing_operations',
       'net_cash_from_investing_activities',
       'long_term_debt_issuances_repayments', 'dividends',
       'other_financing_activities',
       'net_cash_from_financing_activities_continuing_operations',
       'net_cash_from_financing_activities', 'change_in_cash_and_equivalents',
       'effect_of_currency_exchange_rate',
       'sale_of_proper

**- files=12468:** cobertura completa del endpoint.  
**- read_ok=True 12468:** todos legibles.  
**- ticker_col_ok=True 12468:** ticker interno correcto.  
**- dataset_col_ok=True 12468:** _dataset consistente.  
**- ingested_utc_parseable=True 12468:** timestamp bien formado.  
**- rows_business_zero=4445:** sin datos de negocio en ~35.7% de tickers.  
**- tickers_col_mismatch_rows_gt0=0:** sin contradicciones en tickers.  
**- cik_nunique={0:4445,1:7765,2:248,3:10}:**  
    - ideal (1): 7,765  
    - sin CIK (0): 4,445  
    - ambiguo (>1): 258 archivos (248+10)  
**- missing_required_cols={'':8023,'cik':4445}:** coincide con los vacíos.  
**- rows_hit_page_cap=0:** sin truncado por paginación.  

Conclusión de cash_flow_statements: integridad técnica buena, con dos puntos de negocio a vigilar: rows_business_zero y cik_nunique>1  
(258 casos).  

In [58]:
# 4) ratios
print("auditando financials RATIOS")
print()
print("->", p_)
print()
p_ = PATHS["ratios"]
df = pd.read_parquet(p_)
print(df.columns)
# display(df.head(1).T)

# 3) cash_flow_statements
print_dataset_report(d, "ratios")

auditando financials RATIOS

-> D:\financial\cash_flow_statements\ticker=A\cash_flow_statements_A.parquet

Index(['ticker', 'cik', 'date', 'price', 'average_volume', 'market_cap',
       'earnings_per_share', 'price_to_earnings', 'price_to_book',
       'price_to_sales', 'price_to_cash_flow', 'price_to_free_cash_flow',
       'dividend_yield', 'return_on_assets', 'return_on_equity',
       'debt_to_equity', 'current', 'quick', 'cash', 'ev_to_sales',
       'ev_to_ebitda', 'enterprise_value', 'free_cash_flow', '_dataset',
       '_ingested_utc'],
      dtype='str')

=== ratios ===
files: 12468
read_ok: {True: 12468}
ticker_col_ok: {True: 12468}
dataset_col_ok: {True: 12468}
ingested_utc_parseable: {True: 12468}
rows_business_zero: 8159
tickers_col_mismatch_rows_gt0: 0
cik_nunique: {0: 8159, 1: 4307, 2: 2}
missing_required_cols: {'': 12468}
rows_hit_page_cap: 0


**- rows_business_zero=8159:** sin datos de negocio en ~65.4% de tickers (muy alto).  
**- tickers_col_mismatch_rows_gt0=0:** sin contradicciones en tickers.  
**- cik_nunique={0:8159,1:4307,2:2}:**  
    - ideal (1): 4,307  
    - sin CIK (0): 8,159  
    - ambiguo (>1): 2 archivos  
**- missing_required_cols={'':12468}:** sin faltantes requeridos para este endpoint.  
**- rows_hit_page_cap=0:** sin truncado por paginación.  

Conclusión de ratios: integridad técnica buena; el principal punto a vigilar es cobertura útil (rows_business_zero muy alto).

# Fechas inicio y fin de cada tiker en finantials por end point

In [67]:
import pandas as pd
p = r"C:\Users\AlexJ\financial_audit\date_ranges_by_ticker_endpoint.csv"
r = pd.read_csv(p)
print(r[["dataset","ticker","date_col_used","date_start","date_end"]].head(20).to_string(index=False))

#Si quieres ver solo los que sí tienen datos:
print()
print("sí tienen datos")
ok = r[r["date_start"].notna()].copy()
print(ok[["dataset","ticker","date_col_used","date_start","date_end"]].head(30).to_string(index=False))

# Y solo los vacíos:
print()
print("solo los vacío")
nodata = r[r["date_start"].isna()].copy()
print(nodata[["dataset","ticker"]].head(30).to_string(index=False))

          dataset ticker date_col_used                date_start                  date_end
income_statements      A    period_end 2010-01-31 00:00:00+00:00 2026-01-31 00:00:00+00:00
income_statements     AA    period_end 2010-03-31 00:00:00+00:00 2025-12-31 00:00:00+00:00
income_statements    AAA           NaN                       NaN                       NaN
income_statements   AABA           NaN                       NaN                       NaN
income_statements    AAC    period_end 2014-12-31 00:00:00+00:00 2023-06-30 00:00:00+00:00
income_statements   AACB           NaN                       NaN                       NaN
income_statements   AACI           NaN                       NaN                       NaN
income_statements   AACQ    period_end 2020-09-30 00:00:00+00:00 2021-03-31 00:00:00+00:00
income_statements   AACT    period_end 2023-06-30 00:00:00+00:00 2025-06-30 00:00:00+00:00
income_statements   AAGR    period_end 2023-12-31 00:00:00+00:00 2024-03-31 00:00:00+00:00